In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises"

/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises


In [3]:
import sys
sys.path.append("/content/drive/MyDrive/Courses/AI/masked_attention/llama_like")

In [ ]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
  2. Weight tying — parámetros compartidos
"""

'\nTaller: Internals de un LLM estilo LLaMA\n=================================================================\nInstrucciones:\n  - Busca los bloques marcados con TODO y completa el código.\n  - Cada sección tiene una celda de verificación al final.\n  - No modifiques nada fuera de los bloques TODO.\n \nSecciones:\n  1. RMSNorm y SwiGLU desde cero\n'

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [ ]:
# ===========================================================================
# SECCIÓN 2 — Weight tying
# ===========================================================================
# El embedding (vocab → d_model) y el lm_head (d_model → vocab) comparten
# el mismo tensor de pesos. Esto reduce parámetros y mejora generalización.
#
# Pregunta: ¿cuántos parámetros se ahorran con weight tying?
# ===========================================================================

def count_parameters(model: nn.Module) -> int:
    # TODO 2.1 — Retorna el número de parámetros únicos (no contar tensores compartidos)
    # Pista: itera model.parameters() y usa el id() del tensor para evitar duplicados
    # return ...
    raise NotImplementedError("TODO 2.1")


def build_model_no_tying(config: ModelConfig) -> MiniLLaMA:
    # TODO 2.2 — Construye un MiniLLaMA y ROMPE el weight tying
    # Pista: después de construir el modelo, asigna un nuevo nn.Linear a lm_head
    #        con los mismos dims pero pesos independientes
    # model = MiniLLaMA(config)
    # model.lm_head = ...
    # return model
    raise NotImplementedError("TODO 2.2")


# ── Verificación 2 ──────────────────────────────────────────────────────────
def verify_section2():
    print("=" * 55)
    print("VERIFICACIÓN 2 — Weight tying")
    print("=" * 55)
    cfg = ModelConfig(vocab_size=256, d_model=32, n_heads=4,
                      n_kv_heads=2, d_ff=64, max_seq_len=16)

    model_tied    = MiniLLaMA(cfg)
    model_no_tie  = build_model_no_tying(cfg)

    params_tied   = count_parameters(model_tied)
    params_no_tie = count_parameters(model_no_tie)
    saved         = params_no_tie - params_tied
    expected_save = cfg.vocab_size * cfg.d_model

    print(f"  Params con tying    : {params_tied:,}")
    print(f"  Params sin tying    : {params_no_tie:,}")
    print(f"  Diferencia          : {saved:,}")
    print(f"  Esperado (V×D)      : {expected_save:,}")

    tying_ok = (model_tied.lm_head.weight.data_ptr() ==
                model_tied.embed.weight.data_ptr())
    print(f"  Mismo tensor (tied) : {tying_ok}")
    no_tie_ok = (model_no_tie.lm_head.weight.data_ptr() !=
                 model_no_tie.embed.weight.data_ptr())
    print(f"  Tensor distinto (no tied): {no_tie_ok}")

    if saved == expected_save and tying_ok and no_tie_ok:
        print("\n  ✓ Sección 2 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")

In [ ]:
verify_section2()